# Exp 7: HTAP Fresh-Query Latency (total cost: setup + rebuild/apply + probe)

**Scenario**: Base MVCC table exists. N% updates arrive. A query is issued immediately.
How long until the reader receives fresh results?

**Cost breakdown:**
| Approach | Setup | Pre-probe blocking | Probe |
|----------|-------|-------------------|-------|
| SNAP  | O(\|R\|) build + mark_ts(0) | O(\|R\|) mark_ts(1) rebuild — **constant** regardless of N% | O(\|L\|) |
| IVMH  | O(\|R\|) in-place insert | O(N%) in-place apply, **then** probe (serialized) | O(\|L\|) |
| MVHT  | O(\|R\|+overhead) | **0** — reader probes at query_ts concurrently with writer | O(\|L\|) |

**X-axis**: update % of PART &nbsp; **Y-axis**: latency (ms)
**Columns**: `setup_ms` / `rebuild_ms` / `update_ms` / `query_ms` / `total_ms`

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'matplotlib', '--quiet'])
print('done')

In [ ]:
from pathlib import Path
import subprocess, os, sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

ROOT     = Path('../../').resolve()
EXP7_DIR = (ROOT / 'benches' / 'exp7_fresh_query').resolve()
DATA_DIR = EXP7_DIR / 'data'
TPCH_DIR = (ROOT / 'benches' / 'sigmod' / 'tpch_data').resolve()
BIN      = ROOT / 'target' / 'release' / 'exp7_bench'

DATA_DIR.mkdir(parents=True, exist_ok=True)
(EXP7_DIR / 'figs').mkdir(parents=True, exist_ok=True)

# ===== CONFIG =====
SF         = '1.0'
BUCKET_NUM = 2048
WARMUP     = 1
REPEAT     = 5
TRIM       = 1

PART_FILE     = TPCH_DIR / f'part_sf{SF}.tbl'
LINEITEM_FILE = TPCH_DIR / f'lineitem_probe_sf{SF}_1995-09-01_1995-10-01.tbl'

# Update percentage sweep (as fraction, e.g. 0.0001 = 0.01%)
# 0.01% → 0.10% in steps of 0.01%
SWEEP_PCTS = [round(i * 0.0001, 4) for i in range(1, 11)]
# SWEEP_PCTS = [0.0002, 0.0004, 0.0006, 0.0008, 0.001]  # 5-point variant

# Table/repair combos to benchmark
# (table_type, repair_mode, display_label)
SERIES = [
    ('snap',  'nr', 'SNAP'),
    ('ivmh',  'nr', 'IVMH'),
    ('heap',  'wr', 'MONO-WR'),
    ('chain', 'wr', 'DUAL-WR'),
    ('par',   'wr', 'EPOCH-WR'),
]

# Paul Tol Bright palette
tol = {
    'blue':   '#4477AA',
    'cyan':   '#66CCEE',
    'green':  '#228833',
    'yellow': '#CCBB44',
    'red':    '#EE6677',
    'purple': '#AA3377',
    'grey':   '#BBBBBB',
}

STYLE = {
    'SNAP':     (tol['red'],    ':',  'x',  2.0),
    'IVMH':     (tol['yellow'], '--', '+',  2.0),
    'MONO-WR':  (tol['blue'],   '-',  'o',  1.6),
    'DUAL-WR':  (tol['cyan'],   '-',  's',  1.6),
    'EPOCH-WR': (tol['green'],  '-',  'D',  1.6),
}

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['Times New Roman', 'Times', 'Nimbus Roman No9 L', 'DejaVu Serif']
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print('ROOT    :', ROOT)
print('PART    :', PART_FILE)
print('LINEITEM:', LINEITEM_FILE)
print('BIN     :', BIN)
print('Sweep   :', [f'{p*100:.2f}%' for p in SWEEP_PCTS])

In [ ]:
# ── Build Rust binary ──────────────────────────────────────────────────────
print('Building exp7_bench...')
result = subprocess.run(
    ['cargo', 'build', '--release', '--bin', 'exp7_bench'],
    cwd=ROOT, capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])
    raise RuntimeError('cargo build failed')
print('Build OK')

In [ ]:
# ── Verify data files ──────────────────────────────────────────────────────
for f in [PART_FILE, LINEITEM_FILE]:
    if not f.exists():
        raise FileNotFoundError(f'Missing: {f}')
    rows = sum(1 for _ in open(f))
    print(f'  {f.name}: {rows:,} rows')

# Print how many updates each pct translates to
n_parts = sum(1 for _ in open(PART_FILE))
print()
for pct in SWEEP_PCTS:
    n = max(1, round(pct * n_parts))
    print(f'  {pct*100:.2f}% → {n} updates out of {n_parts:,} PART rows')

In [ ]:
# ── Helper: run exp7_bench for one (table, repair, update_pct) point ──────
def run_exp7(table_type, repair_mode, update_pct, output_csv):
    cmd = [
        str(BIN),
        '--part-file',      str(PART_FILE),
        '--lineitem-file',  str(LINEITEM_FILE),
        '--update-pct',     str(update_pct),
        '--table-type',     table_type,
        '--repair-mode',    repair_mode,
        '--bucket-num',     str(BUCKET_NUM),
        '--warmup',         str(WARMUP),
        '--repeat',         str(REPEAT),
        '--trim',           str(TRIM),
        '--output-csv',     str(output_csv),
    ]
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  FAILED [{table_type}/{repair_mode}]: {r.stderr[:500]}')
        return False
    for line in r.stderr.strip().split('\n'):
        if 'ms' in line.lower() or 'pct' in line.lower():
            print(f'  {line.strip()}')
    return True

In [ ]:
# ── Drop old prepare-updates cell (no longer needed: update_pct passed directly) ──
# The bench generates updates from the first N% of PART rows deterministically.
print('No update files to prepare — update_pct passed directly to binary.')

In [ ]:
# ── Exp 7: Sweep update percentage ────────────────────────────────────────
CSV_7 = DATA_DIR / f'exp7_fresh_query_sf{SF}.csv'

if CSV_7.exists():
    CSV_7.unlink()
    print('Removed old CSV')

total_runs = len(SWEEP_PCTS) * len(SERIES)
current    = 0

for pct in SWEEP_PCTS:
    for (ttype, rmode, label) in SERIES:
        current += 1
        print(f'[{current}/{total_runs}] pct={pct*100:.2f}% table={ttype} repair={rmode}')
        run_exp7(ttype, rmode, pct, CSV_7)

print(f'\nDone. Results -> {CSV_7}')

In [ ]:
# ── Load and normalize CSV ─────────────────────────────────────────────────
df = pd.read_csv(CSV_7)

# Normalize column names (bench stores table_type as-is, lowercase)
df['table_type']  = df['table_type'].str.lower()
df['repair_mode'] = df['repair_mode'].str.lower()

# Build label column
label_map = {(t, r): lbl for (t, r, lbl) in SERIES}
df['label'] = df.apply(lambda row: label_map.get((row['table_type'], row['repair_mode']), 'unknown'), axis=1)

print(df.groupby('label')[['update_pct','setup_ms','rebuild_ms','update_ms','query_ms','total_ms']].mean().round(2))
df.head()

In [ ]:
# ── Plot 1: Total latency vs update % (3 sub-panels) ─────────────────────
# Left:  total_ms (full picture including setup)
# Mid:   pre_probe_ms = rebuild_ms + update_ms (blocking before reader starts)
# Right: query_ms (probe time — should be flat for all)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
ax1, ax2, ax3 = axes

x_ticks = [p * 100 for p in SWEEP_PCTS]  # display as %

for (ttype, rmode, label) in SERIES:
    color, ls, marker, lw = STYLE[label]
    rows = df[df['label'] == label].sort_values('update_pct')
    if rows.empty:
        continue
    x = rows['update_pct'] * 100  # convert to %

    # total_ms
    ax1.plot(x, rows['total_ms'], label=label,
             color=color, linestyle=ls, marker=marker, linewidth=lw, markersize=6)

    # pre-probe blocking = rebuild_ms (SNAP) or update_ms (IVMH/MVHT concurrent hidden)
    # For SNAP: blocking = rebuild_ms; For IVMH: blocking = update_ms; For MVHT: 0
    pre_probe = rows['rebuild_ms'] + rows['update_ms']
    ax2.plot(x, pre_probe, label=label,
             color=color, linestyle=ls, marker=marker, linewidth=lw, markersize=6)

    # query_ms (probe time)
    ax3.plot(x, rows['query_ms'], label=label,
             color=color, linestyle=ls, marker=marker, linewidth=lw, markersize=6)

for ax, title, ylabel in [
    (ax1, 'Total latency (setup+wait+probe)', 'Total (ms)'),
    (ax2, 'Pre-probe blocking time',          'Blocking (ms)'),
    (ax3, 'Probe time',                       'Probe (ms)'),
]:
    ax.set_xlabel('Update % of PART')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)
    ax.legend(fontsize=8, framealpha=0.9)

fig.suptitle(f'Exp 7: HTAP Fresh-Query Latency — SF={SF}', fontsize=12)
plt.tight_layout()

out = EXP7_DIR / 'figs' / f'exp7_total_sf{SF}.pdf'
plt.savefig(str(out), format='pdf')
plt.show()
print('Saved:', out)

In [ ]:
# ── Plot 2: Stacked bar breakdown (setup / rebuild / update / query) ───────
# One group per update_pct, one bar per approach.
# Shows clearly what fraction of total latency comes from each component.

pcts_pct = [p * 100 for p in SWEEP_PCTS]
n_pct    = len(SWEEP_PCTS)
labels_in_plot = [lbl for (_, _, lbl) in SERIES]
n_series = len(labels_in_plot)

x = np.arange(n_pct)
width = 0.8 / n_series

fig, ax = plt.subplots(figsize=(max(10, n_pct * 1.5), 5))

for i, (ttype, rmode, label) in enumerate(SERIES):
    color, ls, marker, lw = STYLE[label]
    rows = df[df['label'] == label].sort_values('update_pct')
    if rows.empty:
        continue
    positions = x + (i - n_series / 2 + 0.5) * width

    bottom = np.zeros(n_pct)
    for component, alpha in [('setup_ms', 0.4), ('rebuild_ms', 1.0), ('update_ms', 0.7), ('query_ms', 0.9)]:
        vals = rows[component].values
        ax.bar(positions, vals, width * 0.9, bottom=bottom,
               color=color, alpha=alpha, label=f'{label} {component}' if i == 0 else None)
        bottom += vals

    # Invisible bar for legend entry
    ax.bar(positions[0], 0, width * 0.9, color=color, label=label)

ax.set_xticks(x)
ax.set_xticklabels([f'{p:.2f}%' for p in pcts_pct])
ax.set_xlabel('Update % of PART')
ax.set_ylabel('Latency (ms)')
ax.set_title(f'Exp 7: Latency Breakdown (setup / rebuild / apply / probe) — SF={SF}')
ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color=STYLE[l][0], label=l)
    for (_,_,l) in SERIES
], fontsize=9, loc='upper right')

# Component legend
from matplotlib.patches import Patch
comp_legend = [
    Patch(facecolor='grey', alpha=0.4, label='setup'),
    Patch(facecolor='grey', alpha=1.0, label='rebuild'),
    Patch(facecolor='grey', alpha=0.7, label='apply (in-place/writer)'),
    Patch(facecolor='grey', alpha=0.9, label='probe'),
]
ax.legend(
    handles=[plt.Rectangle((0,0),1,1, color=STYLE[l][0], label=l) for (_,_,l) in SERIES]
    + comp_legend,
    fontsize=8, ncol=2, loc='upper left'
)

plt.tight_layout()
out = EXP7_DIR / 'figs' / f'exp7_breakdown_sf{SF}.pdf'
plt.savefig(str(out), format='pdf')
plt.show()
print('Saved:', out)

# ── Summary table ──────────────────────────────────────────────────────────
print('\nMean total_ms by label across all update %:')
print(df.groupby('label')[['setup_ms','rebuild_ms','update_ms','query_ms','total_ms']]
      .mean().round(3).to_string())